# Bjerknes Diagnostic Lab: ENSO Dynamics from First Principles

> *"Where did pressure stop changing?"* is not the same question as *"Where is the pressure field steepest right now?"*
>
> In certain cold years, they get the same approximate answer. In most years, they do not.

---

## 1. The Thermodynamic Engine and the Geometric Diagnostic

In 1969, Jacob Bjerknes published a landmark paper establishing a fundamental causal chain for global atmospheric teleconnections. He proposed that an anomalously great heat supply from the equatorial ocean intensifies the ascending branch of the Hadley circulation. This intensified overturning circulation subsequently maintains a greater-than-normal flux of angular momentum to the midlatitude westerlies. Ultimately, the strength of the wintertime Hadley circulation is decided by the heat input from the equatorial ocean.

Within this macroscopic thermodynamic framework, Bjerknes noted a specific geographic coincidence regarding the pressure field that drives the equatorial easterlies: in the cold Januaries of 1963, 1965, and 1967, the longitude where the **zero isallobar** crossed the equator was approximately the same as the longitude of the **maximum westward pressure gradient**. 

This notebook asks a simple question about that specific observation: **is that coincidence a mathematical necessity, or an empirical accident of those particular years?**

The answer matters far beyond atmospheric physics. In catastrophe (CAT) modelling, every layer of the risk chain — hazard, exposure, vulnerability, loss — is conditioned on getting the ENSO state right. Not just the categorical label, but the *spatial structure* of the anomaly. If your diagnostic conflates two quantities that only coincide in a low-index regime, the error propagates silently through the entire model.

## 2. Two Questions That Look Like One

Before writing any code, it is worth being precise about what we are separating.

**The zero isallobar** is defined by the *isallobaric change*: $\Delta p(x) = p(x,\, t{+}\Delta t) - p(x,\, t)$. An isallobar is a contour of that field — a line on the map joining points that changed by an equal amount over the interval. The *zero* isallobar is the contour where that change is exactly zero: pressure neither rose nor fell. Where that contour crosses the equator is a statement about **time** — it locates the node of the Southern Oscillation see-saw.

**The gradient maximum** is defined by the *spatial gradient*: $\partial p / \partial x$, evaluated at a single moment. Its peak (most negative value, for the Walker pattern) marks where the pressure slope is steepest. Because the pressure gradients maintaining the northeast trade winds govern the equatorial upwelling, this spatial peak identifies the strongest wind forcing. This is a statement about **space** — it locates the steepest part of the pressure field *right now*, which dictates the suppression or release of the ocean heat that powers the Hadley cell.

These are different derivatives of different functions. The notebook will show that they coincide only in a specific corner of parameter space — and separate, or vanish entirely, everywhere else.

---

## 3. The Idealised Pressure Model

We model the equatorial sea-level pressure as a smooth west-to-east profile using a hyperbolic tangent — high pressure in the west (the maritime continent), low in the east (the cold tongue). This captures the Walker circulation pattern with four interpretable parameters:

$$p(x) = p_{\text{base}} \;-\; A\,\tanh\!\left(\frac{x - x_0}{w}\right)$$

| Parameter | Physical meaning | Controls |
|---|---|---|
| $A$ | Amplitude (half the east–west pressure difference) | **Strength** of the Walker gradient |
| $x_0$ | Inflection longitude | **Location** of the steepest slope = gradient maximum |
| $w$ | Width | How sharp vs. spread-out the slope is |
| $p_{\text{base}}$ | Mean level | A **basin-wide** offset (uniform up or down) |

Two key results follow directly from this functional form:

1. The **spatial gradient** $\partial p/\partial x = -\frac{A}{w}\,\operatorname{sech}^2\!\left(\frac{x-x_0}{w}\right)$ is most negative at $x = x_0$. The gradient maximum sits at $x_0$ by construction — it does not move.

2. The **isallobaric change** $\Delta p(x) = p(x,\,t{+}\Delta t) - p(x,\,t)$ is the difference of two such profiles. Its zero crossings are the longitudes where the zero isallobar meets the equator.

The physics functions are defined in `bjerknes_physics.py` and imported below. This keeps the notebook focused on the *scientific argument* rather than the implementation details.

In [ ]:
# Import the physics module and set up the notebook
from bjerknes_physics import (
    pressure, gradient, find_zero_crossings, get_regime, plot_scenario, DEFAULT_X as X
    plot_scenario, lon_fmt, X
)

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

---

## 4. The Single Knob That Generates Everything

We hold the Walker gradient geometry fixed — the amplitude *intensifies* ($A_1 = 3.5 \to A_2 = 4.2$) about a fixed centre $x_0$ — and vary a single number:

$$\Delta\bar{p} \;=\; \text{basin-wide uniform pressure change between } t \text{ and } t{+}\Delta t$$

Physically, $\Delta\bar{p}$ is the mass-exchange term of the Southern Oscillation: pressure rising or falling uniformly across the entire Pacific, on top of the local gradient steepening. Working the algebra, the isallobaric change becomes:

$$\Delta p(x) \;=\; \Delta\bar{p} \;-\; (A_2 - A_1)\,\tanh\!\left(\frac{x - x_0}{w}\right)$$

The zero isallobar crossing sits where $\tanh\!\left(\frac{x - x_0}{w}\right) = \frac{\Delta\bar{p}}{A_2 - A_1}$.

Since $\tanh$ is bounded between $-1$ and $+1$, the crossing exists only when $|\Delta\bar{p}| < |A_2 - A_1|$. When it exists, its position relative to $x_0$ (the gradient maximum) depends entirely on the sign and magnitude of $\Delta\bar{p}$:

| $\Delta\bar{p}$ | Crossing position | Regime |
|---|---|---|
| $= 0$ | Exactly at $x_0$ | **A** — coincides with gradient max (Bjerknes's cold years) |
| Small positive | East of $x_0$ | **C** — crossing east of max |
| Small negative | West of $x_0$ | **D** — crossing west of max |
| $|\Delta\bar{p}| > A_2 - A_1$ | No crossing | **B** — no equatorial crossing (El Niño onset) |

This is logically complete. Two points on a line can only be arranged four ways: one east of the other, on it, west of it, or absent. There is no fifth scenario.

---

## 5. Scenario A — Cold Year (Crossing Coincides with Gradient Maximum)

The Walker gradient intensifies with **no** basin-wide offset ($\Delta\bar{p} = 0$). The isallobaric change is a clean antisymmetric dipole centred on $x_0$, so its zero sits **exactly on** the gradient maximum.

This is Bjerknes's cold-year diagnostic: the temporal node and the spatial peak give the same longitude. If you only ever study cold years, you will never notice they are different quantities.

In [ ]:
crossings_A, gmax = plot_scenario(
    dpbar=0.0,
    scenario_title="Scenario A — cold year, shift toward central Pacific",
    subtitle="Strong eastward pressure rise; zero isallobar crosses equator near 175°W; "
             "maximum spatial gradient co-located with cold anomaly"
)

**Reading the figure:** In panel ③ the orange dot (zero isallobar) sits at 175°W. In panel ④ the violet dot (gradient maximum) also sits at 175°W. They are vertically aligned — this alignment *is* the cold-year coincidence Bjerknes described.

---

## 6. Scenario B — Warm Year (No Mid-Pacific Crossing)

A large basin-wide pressure **rise** ($\Delta\bar{p} = 1.6$, exceeding $A_2 - A_1 = 0.7$) lifts the entire $\Delta p$ curve above zero. The see-saw node is pushed off the Pacific — **there is no equatorial crossing**.

This is the El Niño onset signature. The Walker circulation has weakened so much that the isallobaric field no longer changes sign within the basin.

In [ ]:
crossings_B, _ = plot_scenario(
    dpbar=1.6,
    scenario_title="Scenario B — warm year (El Niño onset)",
    subtitle="Pressure gradient flattens; zero isallobar shifts eastward or disappears; "
             "weak spatial gradient over central Pacific"
)

**Reading the figure:** Panel ③ shows no orange dot — the green curve stays positive everywhere. The gradient maximum in panel ④ still exists (the Walker pattern hasn't vanished), but the temporal node has left the basin entirely.

If your diagnostic *requires* a zero-isallobar longitude, it will either fail silently or return a spurious value in this regime.

---

## 7. Scenario C — Neutral Year (Crossing East of Gradient Maximum)

A modest basin-wide **rise** ($\Delta\bar{p} = +0.35$) slides the zero crossing **east** of $x_0$. The gradient maximum has not moved — only the crossing has. The temporal node and the spatial peak are now clearly different longitudes, separated by about 12° of longitude.

In [ ]:
crossings_C, _ = plot_scenario(
    dpbar=0.35,
    scenario_title="Scenario C — neutral year, isallobar shifts east",
    subtitle="Pressure gradient moderate; zero isallobar crosses east of 160°W; "
             "gradient maximum and isallobar crossing are clearly separated"
)

---

## 8. Scenario D — Neutral Year (Crossing West of Gradient Maximum)

A modest basin-wide **fall** ($\Delta\bar{p} = -0.35$) slides the zero crossing **west** of $x_0$, to the far side of the dateline. This is the mirror image of Scenario C — the crossing now sits *west* of the steepest slope.

In [ ]:
crossings_D, _ = plot_scenario(
    dpbar=-0.35,
    scenario_title="Scenario D — neutral year, isallobar shifts west",
    subtitle="Pressure gradient moderate; zero isallobar crosses west of dateline; "
             "gradient maximum well east of the crossing"
)

**The C ↔ D pair** makes the separation visible in both directions. In C the crossing is displaced east; in D it is displaced west. Neither is pathological — both are ordinary neutral-year configurations. Yet in both, a diagnostic that assumes coincidence would be wrong by 12° of longitude or more.

---

## 9. Are There More Scenarios?

The sweep below makes the completeness argument visual. As $\Delta\bar{p}$ runs from negative to positive, the crossing migrates continuously from west of the gradient maximum, through it (at $\Delta\bar{p} = 0$), to east of it — and **disappears** entirely once $|\Delta\bar{p}|$ exceeds the gradient change $A_2 - A_1 = 0.7$.

Four regions, no fifth.

In [ ]:
from bjerknes_physics import pressure as P, DEFAULT_X as X , find_zero_crossings as fzc

A1, A2, x0, w, base = 3.5, 4.2, 185.0, 22.0, 1010.0
dA = A2 - A1
dps = np.linspace(-1.2, 1.2, 600)
xc = []
for d in dps:
    dp = P(X, A2, x0, w, base + d) - P(X, A1, x0, w, base)
    cs = fzc(X, dp)
    xc.append(cs[0] if cs else np.nan)
xc = np.array(xc)

from bjerknes_physics import LON_TICKS, LON_LABELS, lon_fmt, DEFAULT_X as X

plt.figure(figsize=(9.5, 5))
plt.axvspan(-1.2, -dA, color="grey",   alpha=0.15, label="B: no crossing (negative)")
plt.axvspan(-dA,   0.0, color="orange", alpha=0.10, label="D: crossing west of max")
plt.axvspan( 0.0,   dA, color="teal",   alpha=0.10, label="C: crossing east of max")
plt.axvspan( dA,   1.2, color="grey",   alpha=0.15, label="B: no crossing (positive)")
plt.axhline(x0, color="violet", lw=2.5, label="gradient max (fixed)")
plt.plot(dps, xc, color="darkorange", lw=2.5, label="zero-isallobar crossing")
plt.axvline(0, color="k", lw=0.8, ls=":")
plt.scatter([0], [x0], color="red", zorder=6, s=80, label=r"A: coincide ($\Delta\bar{p}=0$)")
yt = [150, 165, 180, 195, 210, 225]
plt.yticks(yt, [lon_fmt(v) for v in yt])
plt.xlabel(r"$\Delta\bar{p}$  (basin-wide pressure change, hPa)", fontsize=11)
plt.ylabel("Crossing longitude", fontsize=11)
plt.title("One knob, four regimes", fontsize=13, fontweight="bold")
plt.legend(fontsize=8, loc="upper left", ncol=2)
plt.grid(alpha=0.15)
plt.tight_layout()
plt.show()
print(f"Gradient change A2−A1 = {dA:.2f} hPa → crossing exists only while |Δp̄| < {dA:.2f}")

The orange curve traces the crossing longitude as a function of $\Delta\bar{p}$. It passes through the violet line (gradient maximum) exactly once, at $\Delta\bar{p} = 0$, and disappears at the grey boundaries. Every real ENSO state maps to a point on or off this curve.

---

## 10. Automated Sanity Check

The classifier below takes any pair of pressure profiles and decides which regime you are in — the automated version of the visual judgement above.

In [ ]:
print("Regime classification for all four scenarios:\n")
for name, dpbar in [("A cold", 0.0), ("B warm", 1.6),
                     ("C neutral-east", 0.35), ("D neutral-west", -0.35)]:
    p1 = pressure(X, 3.5, 185.0, 22.0, 1010.0)
    p2 = pressure(X, 4.2, 185.0, 22.0, 1010.0 + dpbar)
    dp = p2 - p1
    cs = find_zero_crossings(X, dp)
    regime = get_regime(dpbar, 3.5, 4.2)
    if cs:
        rel = "ON" if abs(cs[0] - 185.0) < 0.5 else (
              "EAST of" if cs[0] > 185.0 else "WEST of")
        print(f"  {name:18s}  Δp̄={dpbar:+.2f}  →  {regime:22s}  "
              f"crossing {lon_fmt(cs[0]):>6s}  ({rel} max)")
    else:
        print(f"  {name:18s}  Δp̄={dpbar:+.2f}  →  {regime:22s}  "
              f"no crossing")

In [1]:
import os
print(os.getcwd())

/Users/dantsai/Documents/GitHub/Bjerknes-Diagnostic-Lab


---

## 11. Discussion: Why This Matters Beyond the Figure

### 11.1 The Nature of the Error

The four panels exist to keep two questions apart: *where did pressure stop changing* (the zero isallobar — a temporal quantity) versus *where is the field steepest right now* (the gradient maximum — a spatial quantity). The notebook has shown they coincide only in the cold-year corner of parameter space ($\Delta\bar{p} \approx 0$) and separate — or vanish entirely — everywhere else.

Critically, the error is **regime-dependent**. It is not random noise that averages out across a stochastic catalogue. It is a patterned bias:

- **Precise where it happens to be easy** (cold years, where the two coincide)
- **Wrong by a fixed displacement** in neutral years (C and D, in opposite directions)
- **Confidently asserting a crossing that does not exist** in warm years (B)

### 11.2 Downstream Consequences for CAT Modelling

In a catastrophe model, the ENSO state—as determined by the diagnostic or index chosen to classify equatorial Pacific conditions—conditions every layer of the risk chain. Whether that state comes from the zero isallobar, the gradient maximum, an ENSO index, or reanalysis-derived phase classification, the choice of diagnostic upstream propagates downstream. If the diagnostic conflates temporal and spatial properties (as the zero isallobar confusion does in most ENSO states), that misclassification is inherited by every risk layer.

**Hazard.** Tropical cyclone genesis corridors, track density distributions, and basin-to-landfall return periods are all fit per ENSO phase. A mislocated gradient-maximum longitude shifts the genesis corridor — and the shift is systematic, not random. If the historical record is phase-labelled on the wrong diagnostic, the conditional genesis probabilities are fitted on a contaminated sample.

**Exposure.** Flood and wind footprints follow the displaced hazard. Assets that are actually exposed get missed; others get over-exposed. The error is spatial, so it cannot be fixed by adjusting a scalar loading factor.

**Vulnerability.** Historical loss events get phase-labelled on the wrong ENSO state. Depth–damage functions and wind-loss curves are then fitted on a contaminated sample — cold-year and warm-year events pooled together without anyone seeing the boundary.

**Loss.** The annual average loss (AAL) for Pacific perils carries an ENSO-conditional frequency term. A systematic phase-classification error contaminates this term across every simulated year of a stochastic catalogue.

**Resilience.** Adaptation investment — where you harden infrastructure, where you set early-warning thresholds, which communities get priority in a multi-hazard risk-reduction strategy — is calibrated off the hazard model. If the spatial footprint is wrong, resilience spending is geolocated incorrectly.


### 11.3 The Practical Rule

A **uniform** error (same offset everywhere) can be calibrated out once discovered. A **regime-dependent** error has to be *found first* — and in the reinsurance industry, "harder to find" means "longer to be wrong, and longer to be pricing incorrectly."

The clean takeaway: asking *"where did pressure stop changing?"* and asking *"where is the pressure field steepest right now?"* are not the same question. If your ENSO diagnostic conflates them, you are not working with an approximation — you are answering a different question than the one you think you are asking, and every model layer that inherits that answer is wrong in a structured, non-random way.

### 11.4 Limitations of This Notebook

This notebook uses an idealised $\tanh$ profile with a single control parameter. Real equatorial pressure fields are shaped by multiple processes simultaneously (trade-wind variability, off-equatorial forcing, MJO modulation). A real pressure field could have a more complex $\Delta p$ profile than a shifted tanh, potentially producing multiple zero crossings or non-monotonic gradients. The notebook demonstrates the *principle* — that the temporal node and the spatial peak are mathematically independent objects — not the full complexity of any particular year.

Extending this to reanalysis data (ERA5 equatorial SLP profiles, 1950–present) would allow testing whether the four regimes map cleanly onto observed ENSO states, and whether the crossing–gradient separation correlates with downstream TC statistics. That is the natural next step.

---

## References

- Bjerknes, J. (1969). *Atmospheric Teleconnections from the Equatorial Pacific.* Monthly Weather Review, **97**(3), 163–172. [DOI:10.1175/1520-0493(1969)097<0163:ATFTEP>2.3.CO;2](https://doi.org/10.1175/1520-0493(1969)097%3C0163:ATFTEP%3E2.3.CO;2)
- Berlage, H. P. (1957). *Fluctuations of the General Atmospheric Circulation of More Than One Year.* Mededelingen en Verhandelingen, No. 69, KNMI.
- Walker, G. T. (1924). *Correlation in Seasonal Variations of Weather, IX.* Memoirs of the India Meteorological Department, 24(9), 275–332.

---

*Notebook: Bjerknes Diagnostic Lab | Author: Shun-Chan Tsai | Last updated: June 2026*

Overwriting bjerknes_physics.py
